# Stochastic Simulation — Exercises & Methods

## A. Bayesian Posterior Derivations

**When**: Derive the posterior distribution given prior and likelihood.

### Recipe — Gaussian Conjugacy (1D)
1. Write p(x|y) ∝ p(y|x)p(x)
2. Expand exponential: collect all terms involving x
3. Group as −½(ax² − 2bx + c) where a = 1/σ² + 1/σ₀²
4. Complete the square: a(x − b/a)² + (c − b²/a)
5. Read off: μ_p = b/a, σ_p² = 1/a

### Recipe — Multivariate Gaussian
1. Write p(x|y) ∝ exp{−½[x^TΛx − 2x^Tt + const]} where Λ = A^TΣ⁻¹A + Σ₀⁻¹, t = A^TΣ⁻¹y + Σ₀⁻¹μ₀
2. Complete the square: (x − Λ⁻¹t)^TΛ(x − Λ⁻¹t)
3. Posterior: N(Λ⁻¹t, Λ⁻¹)

### Recipe — Non-Gaussian Conjugates
- Poisson-Gamma: p(x|y) ∝ x^{α−1+y}e^{−(β+1)x} → Gamma(α+y, β+1)
- Pattern: recognise kernel of known distribution

### Common Exam Tasks
- Derive 1D Gaussian posterior with multiple observations: σ_p² = (n/σ² + 1/σ₀²)⁻¹, μ_p = σ_p²(nȳ/σ² + μ₀/σ₀²)
- Derive marginal likelihood p(y) by integrating out x
- Model comparison: compute p₀(y) vs p₁(y), larger wins

## B. Inverse Transform Sampling

**When**: p* has a CDF with closed-form inverse.

### Recipe
1. Compute CDF: F(x) = ∫_{−∞}^x p*(t)dt
2. Invert: solve U = F(x) for x = F⁻¹(U)
3. Algorithm: U ~ Unif(0,1), X = F⁻¹(U)

### Discrete Version
1. Compute cumulative weights: F(s_k) = Σ_{j=1}^k w_j
2. Draw U ~ Unif(0,1)
3. Find smallest k such that F(s_k) ≥ U

### Key Derivations to Know

| Target | F(x) | F⁻¹(u) |
|--------|-------|--------|
| Exp(λ) | 1−e^{−λx} | −ln(1−u)/λ |
| Cauchy | ½ + arctan(x)/π | tan(π(u−½)) |
| Weibull(k) | 1−e^{−x^k} | (−ln(1−u))^{1/k} |
| Unif(a,b) | (x−a)/(b−a) | a + u(b−a) |

### When Inverse Transform Fails
- Gaussian: CDF has no closed-form inverse → use Box-Müller instead
- Complex distributions → use rejection sampling

## C. Transformation of Random Variables

**When**: Prove that Y = g(X) has a specific distribution.

### Recipe — 1D
1. Given X ~ p_X, Y = g(X)
2. Find inverse: x = g⁻¹(y)
3. Compute derivative: |dg⁻¹/dy|
4. Apply formula: p_Y(y) = p_X(g⁻¹(y)) |dg⁻¹/dy|
5. Recognise the resulting density

### Recipe — 2D (and higher)
1. Given (X₁,X₂) ~ p, (Y₁,Y₂) = g(X₁,X₂)
2. Find inverse: (x₁,x₂) = g⁻¹(y₁,y₂)
3. Compute Jacobian matrix J_{g⁻¹} (matrix of partial derivatives)
4. Compute |det J_{g⁻¹}|
5. Apply: p_{Y₁,Y₂} = p_{X₁,X₂}(g⁻¹(y)) |det J_{g⁻¹}|

### Box-Müller Proof Walkthrough
- X₁ ~ Exp(1/2), X₂ ~ Unif(−π,π)
- g: Y₁ = √X₁ cos X₂, Y₂ = √X₁ sin X₂
- g⁻¹: x₁ = y₁²+y₂², x₂ = arctan(y₂/y₁)
- J = [[2y₁, 2y₂], [−y₂/(y₁²+y₂²), y₁/(y₁²+y₂²)]]
- |det J| = 2
- p_{Y₁,Y₂} = (1/2)e^{−(y₁²+y₂²)/2} × (1/2π) × 2 = N(y₁;0,1)N(y₂;0,1) ✓

### Disk Sampling (Example 2.6)
- r ~ Unif(0,1), θ ~ Unif(−π,π), set X₁ = √r cosθ, X₂ = √r sinθ
- Same Jacobian |det J| = 2
- Result: p(x₁,x₂) = Unif(x₁²+x₂²; 0,1) × (1/2π) × 2 = 1/π for x₁²+x₂² < 1
- **Warning**: without √r, you get p ∝ 1/√(x₁²+x₂²) — NOT uniform!

## D. Rejection Sampling Design

**When**: Sample from p*(x) (or p̄*(x)) using proposal q(x).

### Recipe — Basic Rejection Sampler
1. Choose proposal q(x) that is easy to sample from
2. Find M such that p̄*(x) ≤ Mq(x) for all x:
   - Compute R(x) = p̄*(x)/q(x)
   - Find x* = argmax R(x): take log, differentiate, set to zero, check 2nd derivative
   - M = R(x*)
3. Run Algorithm 5/6: sample X'~q, U~Unif, accept if U ≤ p̄*(X')/(Mq(X'))
4. Acceptance rate: â = 1/M (normalised) or Z/M (unnormalised)

### Recipe — Optimising the Proposal
1. Parameterise q_θ(x)
2. For each θ: compute M_θ = sup_x p̄*(x)/q_θ(x)
3. Minimise: θ* = argmin_θ log M_θ
4. Take derivative of log M w.r.t. θ, set to zero

### Example Walkthrough: Gamma(α,1) target, Exp(λ) proposal
1. R(x) = x^{α−1}e^{(λ−1)x} / (λΓ(α))
2. log R: (α−1)log x + (λ−1)x − log λ − log Γ(α)
3. d/dx log R = (α−1)/x + (λ−1) = 0 → x* = (α−1)/(1−λ)
4. Compute M(λ) from x*
5. Minimise log M w.r.t. λ → λ* = 1/α
6. Final M* = e^{−(α−1)}/Γ(α)

### Checklist for Rejection Problems
- [ ] Is p̄* given or p*? (unnormalised vs normalised)
- [ ] Does Mq(x) ≥ p̄*(x) for ALL x? (check boundaries too)
- [ ] Is M the smallest possible? (optimised x*)
- [ ] Can you further optimise q by tuning parameters?

## E. Rejection Sampling for Bayesian Posteriors

**When**: Sample from p(x|y) without deriving the closed-form posterior.

### Recipe
1. Unnormalised posterior: p̄(x|y) = p(y|x)p(x)
2. Choose proposal q(x) (e.g. the prior, or a Gaussian)
3. Find M: sup_x p(y|x)p(x) / q(x)
4. Rejection algorithm with p̄(x|y) / (Mq(x))

### Example 2.13 — Gaussian Posterior via Rejection
- Prior N(0,1), likelihood N(y;x,σ²), observe y
- p̄(x|y) = N(y;x,σ²)N(x;0,1)
- Proposal: q(x) = N(x;0,1) (the prior itself!), M = 1
- Accept X' if U ≤ N(y;X',σ²) / sup_x N(y;x,σ²)

### Example 2.16 — Poisson-Gamma Posterior via Rejection
- p̄(x|y) = x^{α−1+y}e^{−2x}
- Proposal: Exp(λ), optimise λ* = 2/(α+y)
- Find x*, compute M, run rejection

### Key Insight
Rejection sampling with unnormalised density: don't need to know Z = p(y)!

## F. Composition & Marginalisation

**When**: Sample from joint, marginal, or mixture distributions.

### Joint Sampling Recipe
1. Factorise: p(x₁,…,x_n) = p(x₁)p(x₂|x₁)…p(x_n|x₁,…,x_{n−1})
2. Sample sequentially: X₁ ~ p(x₁), then X₂|X₁, etc.

### Marginal Sampling Recipe
1. Sample from joint: (X₁,X₂) ~ p(x₁,x₂)
2. Keep only X₁ → automatically distributed as p(x₁) = ∫p(x₁,x₂)dx₂

### Mixture Sampling Recipe
1. p*(x) = Σ_k w_k q_k(x)
2. Sample component: k ~ Categorical(w₁,…,w_K)
3. Sample value: X ~ q_k(x)

### Beta from Gamma (Exercise 2.8)
If X₁ ~ Gamma(α,1), X₂ ~ Gamma(β,1), then Y = X₁/(X₁+X₂) ~ Beta(α,β)

## G. Proving Key Results

| Result | Proof Strategy |
|--------|---------------|
| Thm 2.1 (Prob. integral transform) | One line: P(F⁻¹(U)≤x) = P(U≤F(x)) = F(x) |
| Thm 2.2 (Fund. thm of simulation) | x-marginal of uniform on A: q(x) = ∫₀^{p̄*} (1/Z)dy = p̄*/Z = p* |
| Prop 2.3 (Acceptance rate) | â = E[a(X')] = ∫(p̄*/(Mq))q dx = Z/M |
| Box-Müller | 2D transformation of RVs formula; compute g⁻¹ and |det J| = 2 |
| Prop 2.2 (Marginalisation) | P(Z∈A) = ∫_{ℝ^{d₁}×A} p(x₁,x₂) dx₁dx₂ = ∫_A p_{X₂}(x₂)dx₂ |
| Curse of dimensionality | supM = K^d when p*=∏p₀, q=∏q₀; â = 1/K^d → 0 |
| Gaussian marginal likelihood | Complete the square and evaluate Gaussian integral |
| Conditional Bayes rule (Prop 1.1) | Apply standard Bayes rule with z fixed throughout |

## H. Key Formulae Cheat Sheet

### Bayesian Inference
- Bayes: p(x|y) = p(y|x)p(x)/p(y)
- Gaussian (1D): μ_p = (σ²μ₀+σ₀²y)/(σ²+σ₀²), σ_p² = σ²σ₀²/(σ²+σ₀²)
- Gaussian (nD): Σ_p = (A^TΣ⁻¹A+Σ₀⁻¹)⁻¹, μ_p = Σ_p(A^TΣ⁻¹y+Σ₀⁻¹μ₀)
- Marginal likelihood (Gaussian): p(y) = N(y; μ₀, σ₀²+σ²)

### Transformation of RVs
- 1D: p_Y(y) = p_X(g⁻¹(y))|dg⁻¹/dy|
- nD: p_Y(y) = p_X(g⁻¹(y))|det J_{g⁻¹}|

### Sampling Algorithms
- Inverse transform: U ~ Unif, X = F⁻¹(U)
- Box-Müller: Z₁ = √(−2ln U₁)cos(2πU₂), Z₂ = √(−2ln U₁)sin(2πU₂)
- Multivariate Gaussian: Y = μ + Lv, Σ = LL^T
- Rejection: accept if U ≤ p̄*(X')/(Mq(X'))

### Rejection Sampling
- Optimal M: M* = sup_x p̄*(x)/q(x)
- Acceptance rate: â = 1/M (normalised), â = Z/M (unnormalised)
- Curse of dim.: â_d = π^{d/2}/Γ(d/2+1); for product: â = 1/K^d
- Expected cost per accepted sample: O(1/â)

### Useful Distributions
- Exp(λ): p(x) = λe^{−λx}, F(x) = 1−e^{−λx}, F⁻¹(u) = −ln(1−u)/λ
- Gamma(α,β): p(x) = β^α/Γ(α) x^{α−1}e^{−βx}
- Beta(α,β): p(x) = Γ(α+β)/(Γ(α)Γ(β)) x^{α−1}(1−x)^{β−1}
- Poisson(λ): P(k) = λ^k e^{−λ}/k!
- Weibull(k): p(x) = kx^{k−1}exp(−x^k), F(x) = 1−exp(−x^k)

## I. Exam Strategy

### Likely Question Types
1. **Derive posterior** for given prior/likelihood (completing the square for Gaussian; recognise conjugate kernels)
2. **Derive inverse CDF** and write sampling algorithm (Thm 2.1)
3. **Prove transformation of RVs** (compute g⁻¹ and Jacobian; Box-Müller is a classic)
4. **Design rejection sampler**: find M, compute acceptance rate, optimise proposal
5. **Bayesian rejection sampling**: sample posterior without closed-form derivation
6. **Curse of dimensionality**: explain/prove why rejection fails in high-d
7. **Marginal likelihood** computation and model comparison

### Common Mistakes to Avoid
- Forgetting absolute value in transformation formula (|dg⁻¹/dy| or |det J|)
- Using p* instead of p̄* (or vice versa) in rejection — be clear about normalised vs unnormalised
- Not checking that Mq(x) ≥ p̄*(x) for ALL x (including boundaries/tails)
- For disk sampling: forgetting the √r correction
- Confusing acceptance ratio a(x) with acceptance rate â = E[a(X')]

### Time Management
- Bayesian derivations are mechanical (expand, complete square) — do them quickly
- Rejection optimisation: differentiate log R, find x*, substitute — formulaic
- Proofs of Thm 2.1 and 2.2 are very short — write them out fully for easy marks
- For transformation problems: draw the diagram (input RVs → transform → output), compute g⁻¹ systematically